In [ ]:
# This notebook lives in examples/: run it from the repository root so ./Data, ./Figures
# and `from utils import ...` resolve as before.
import os
if os.path.basename(os.getcwd()) == "examples":
    os.chdir("..")

In [1]:
import numpy as np
import networkx as nx
from docplex.mp.model import Model
from collections import defaultdict


# Fully connected problems

In [2]:
def WMaxCut(G):
    # Weighted maxcut problem from networkx graph model as a QUBO problem
    mdl = Model('MaxCut')
    x = {i: mdl.binary_var(name=f"x_{i}") for i in sorted(G.nodes())}
    mdl.minimize(-mdl.sum(G[i][j]["weight"]*(x[i] * (1 - x[j]) + x[j] * (1 - x[i]))  for (i, j) in G.edges))
    return mdl

def WMaxCut(G):
    # Weighted maxcut problem from networkx graph model as a QUBO problem
    mdl = Model('MaxCut')
    num_vertices = G.number_of_nodes()
    translate = {j:i for i, j in enumerate(sorted(G.nodes))}
    x = {i: mdl.binary_var(name=f"x_{i}") for i in range(num_vertices)}
    mdl.minimize(-mdl.sum(G[i][j]["weight"]*(x[translate[i]] * (1 - x[translate[j]]) + x[translate[j]] * (1 - x[translate[i]]))  for (i, j) in G.edges))
    return mdl

In [11]:
np.random.seed(1)
problems = np.load("./Data/WMC_FC.npy", allow_pickle=True).item()

num_qubits = [93]
for nq in num_qubits:
    print(f"Number of qubits: {nq}")
    G  = nx.Graph()
    G.add_nodes_from(range(nq))
    for i in range(nq):
        for j in range(i + 1, nq):
            w = np.random.choice([0.1, 0.2, 0.5, 2.0])
            G.add_weighted_edges_from([[i,j,w]])
    problems[nq]["G"] = G
    mdl = WMaxCut(G)
    mdl.solve() # Solving using CPLEX
    problems[nq]["time_cplex"] = mdl.get_solve_details().deterministic_time
    problems[nq]["real_time_cplex"] = mdl.get_solve_details().time
    problems[nq]["n_iterations_cplex"] = mdl.get_solve_details().nb_iterations
    problems[nq]["status"] = mdl.get_solve_details().status
    problems[nq]["de"] = mdl.get_solve_details().to_string()
    problems[nq]["sol"] = "".join(str(round(mdl.solution.get_value(var))) for var in mdl.iter_binary_vars())
np.save("./Data/WMC_FC.npy", problems)

Number of qubits: 93


# Native Layout Graph

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
service = QiskitRuntimeService(name="mcm-primitives")

In [4]:
backend_name = "ibm_miami"
backend = service.backend(backend_name)

In [5]:
np.random.seed(123)
n_qubits = backend.num_qubits
problem = {}
G = nx.Graph()
G.add_nodes_from(range(n_qubits))
for i,j in backend.coupling_map.get_edges(): 
    w = np.random.choice([0.1, 0.2, 0.3, 0.5, 1.0])
    G.add_weighted_edges_from([[i, j, w]])

mdl = WMaxCut(G)
mdl.solve()
problem["G"] = G
problem["sol"] = "".join(str(round(mdl.solution.get_value(var))) for var in mdl.iter_binary_vars())


problems = np.load("./Data/NL-problems.npy", allow_pickle=True).item()
problems["nighthawk-r1"] = problem
np.save("./Data/NL-problems.npy", problems)
# np.save(f"./Data/WMC_NL_{backend}.npy", problem)

### AWS devices

In [4]:
from braket.aws import AwsDevice, AwsQuantumTask
from braket.devices import Devices, LocalSimulator
from braket.circuits import Circuit
backends = {}
backends["iqm_garnet"] = AwsDevice(Devices.IQM.Garnet)
backends["rigetti_ankaa_3"] = AwsDevice(Devices.Rigetti.Ankaa3)


In [20]:
problems = np.load("./Data/NL-problems.npy", allow_pickle=True).item()
np.random.seed(123)
backend_name = "rigetti_ankaa_3"
problem = {}
G_backend = backends[backend_name].topology_graph
n_qubits = G_backend.number_of_nodes()

G = nx.Graph()
dict_vars = {j:i for i, j in enumerate(sorted(G_backend.nodes))}
G.add_nodes_from(sorted(G_backend.nodes()))
for i,j in G_backend.edges(): 
    w = np.random.choice([0.1, 0.2, 0.3, 0.5, 1.0])
    G.add_weighted_edges_from([[i, j, w]])

mdl = WMaxCut(G)
mdl.solve()
problem["G"] = G
problem["sol"] = "".join(str(round(mdl.solution.get_value(var))) for var in mdl.iter_binary_vars())
problems[backend_name] = problem
np.save("./Data/NL-problems.npy", problems)

NameError: name 'backends' is not defined

In [ ]:
import json
problems = np.load("./Data/NL-problems.npy", allow_pickle=True).item()
np.random.seed(123)
backend_name = "iqm_emerald"
problem = {}
n_qubits = 54

with open(f"./Data/{backend_name}/native_layout.json", "r") as file:
    edges = json.load(file)
G = nx.Graph()
G.add_nodes_from(range(n_qubits))
for i,j in edges: 
    w = np.random.choice([0.1, 0.2, 0.3, 0.5, 1.0])
    G.add_weighted_edges_from([[i, j, w]])

mdl = WMaxCut(G)
mdl.solve()
problem["G"] = G
problem["sol"] = "".join(str(round(mdl.solution.get_value(var))) for var in mdl.iter_binary_vars())
problems[backend_name] = problem
np.save("./Data/NL-problems.npy", problems)

# 1D-Chain Graph

In [1]:
qubits_in_line = {}
qubits_in_line["Eagle"] = list(range(13,-1,-1)) + [14] + list(range(18,33)) + [36] + list(range(51,36,-1)) + [52] + list(range(56,71)) + [74] + list(range(89,74,-1)) + [90] + list(range(94,109)) + [112] + list(range(126,112,-1))
qubits_in_line["Heron-r1"] = list(range(13,-1,-1)) + [15] + list(range(19,34)) + [37] + list(range(52,37,-1)) + [53] + list(range(57,72)) + [75] + list(range(90,75,-1)) + [91] + list(range(95,110)) + [113] + list(range(128,114,-1))
qubits_in_line["Heron-r2"] = list(range(0,16)) + [19] + list(range(35,20,-1)) + [36] + list(range(41,56)) + [59] + list(range(75,60,-1)) + [76] + list(range(81,96)) + [99] + list(range(115,100,-1)) + [116] + list(range(121,136)) + [139] + list(reversed(range(140,156)))

In [ ]:
backend_name = "Heron-r2"
np.random.seed(123)
num_qubits = range(110, 151)
problems = np.load("./Data/problems_1DChain.npy", allow_pickle=True).item()
for nq in num_qubits:
    print(f"Number of qubits: {nq}")
    G = nx.Graph()
    for i in range(nq-1):
        w = np.random.choice([0.1, 0.2, 0.3, 0.5, 1.0])
        G.add_weighted_edges_from([[i, i+1, w]])
    problems["G"][nq] = G
    mdl = WMaxCut(G)
    mdl.solve()
    problems["sol"][nq] = "".join(str(round(mdl.solution.get_value(var))) for var in mdl.iter_binary_vars())
np.save(f"./Data/problems_1DChain.npy", problems)

Number of qubits: 110
Number of qubits: 111
Number of qubits: 112
Number of qubits: 113
Number of qubits: 114
Number of qubits: 115
Number of qubits: 116
Number of qubits: 117
Number of qubits: 118
Number of qubits: 119
Number of qubits: 120
Number of qubits: 121
Number of qubits: 122
Number of qubits: 123
Number of qubits: 124
Number of qubits: 125
Number of qubits: 126
Number of qubits: 127
Number of qubits: 128
Number of qubits: 129
Number of qubits: 130
Number of qubits: 131
Number of qubits: 132
Number of qubits: 133
Number of qubits: 134
Number of qubits: 135
Number of qubits: 136
Number of qubits: 137
Number of qubits: 138
Number of qubits: 139
Number of qubits: 140
Number of qubits: 141
Number of qubits: 142
Number of qubits: 143
Number of qubits: 144
Number of qubits: 145
Number of qubits: 146
Number of qubits: 147
Number of qubits: 148
Number of qubits: 149
Number of qubits: 150


KeyError: 150

# Removing mdl conflicting data

In [12]:
import os 
import numpy as np
folders = [f for f in os.listdir("./Data/") if os.path.isdir(os.path.join("./Data/", f)) and not f.startswith('.')]
for sub_folder in folders:
    folder_path = f"./Data/{sub_folder}/"
    files = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, f)) and not f.startswith('.')]

    # nq = 156
    # backend_name = "ibm_fez"
    # layout = "HE"
    # extra = "_fractional"
    for file in files:
        if not os.path.isdir(os.path.join(folder_path, file)) and not file.startswith('.'):
            data = np.load(folder_path+file, allow_pickle=True).item()
            try:
                data.pop("mdl")
            except:
                pass
            np.save(folder_path+file, data)

In [9]:
import json

def save_ising_file(nq, hs, Js, optinal_sol, filename):
    with open(filename, 'w') as isingfile:
        # Write header lines (required by HPC software)
        mapping = {i:i for i in range(nq)}
        isingfile.write(f'# QUBITS {nq}\n')
        isingfile.write(f'# OFFSET {0}\n')
        isingfile.write(f'# QUBITMAP {mapping}\n')
        isingfile.write(f'# START_METADATA\n')
        isingfile.write(f'# cplex_sol:{optinal_sol}\n')
        isingfile.write(f'# END_METADATA\n')

        # Write Ising values
        for i in sorted(hs):
            isingfile.write(f'{i} {i} {hs[i]}\n')
        for i,j in sorted(Js):
            isingfile.write(f'{i} {j} {Js[i,j]}\n')




In [10]:
import numpy as np
import os

nqs = [40, 44, 48, 50, 56]

data = np.load("./Data/WMC_FC.npy", allow_pickle=True).item()

for nq in nqs:
    G = data[nq]["G"]
    sol = data[nq]["sol"]
    hs = {i: 0.0 for i in G.nodes()}
    Js = {(i,j): G[i][j]["weight"] for (i,j) in G.edges()}
    os.makedirs(f"./Data/JBS/", exist_ok=True)
    save_ising_file(nq, hs, Js, sol, f"./Data/JBS/WMC_FC_{nq}.ising")